### Load Data and Define Features

First, we'll load your CSV file into a pandas DataFrame. Please update the `file_path` variable with the actual path to your CSV file.

Then, you'll need to define the `selected_features` which are the columns you want to use to identify exact duplicate instances. Also, specify the `match_outcome_column` and its corresponding values for 'Home Win', 'Draw', and 'Away Win'.

In [ ]:
import pandas as pd

# --- User Input Required ---
# 1. Update the path to your CSV file
file_path = pd.read_csv('All_Data_Final.csv')  # e.g., '/content/match_data.csv'

# 2. Specify the features that define a unique 'instance' for duplication check
#    e.g., ['team_home', 'team_away', 'season', 'match_day']
selected_features = []

# 3. Specify the column containing the match outcome and its values
#    e.g., 'full_time_result'
match_outcome_column = ''

# Map the outcome values from your dataset to 'H', 'D', 'A'
# e.g., {'H': 'Home Win', 'D': 'Draw', 'A': 'Away Win'}
outcome_mapping = {
    'H': 'Home Win',
    'D': 'Draw',
    'A': 'Away Win'
}
# --------------------------

# Load the dataset
try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}. Shape: {df.shape}")
    print("First 5 rows of the DataFrame:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred while loading the CSV: {e}")

In [ ]:
import pandas as pd
import numpy as np

def prepare_binned_features(df):
    """Bins the features according to the specified constraints."""
    df_binned = df.copy()

    # Score diff: range 4 or more, down to -4 or less
    if 'score_diff' in df_binned.columns:
        df_binned['score_diff_bin'] = df_binned['score_diff'].clip(lower=-4, upper=4)

    # Subs: range 0, 1, 2 (assuming 2 means 2 or more)
    sub_cols = ['home_attk_sub', 'home_def_sub', 'away_attk_sub', 'away_def_sub']
    for col in sub_cols:
        if col in df_binned.columns:
            df_binned[f'{col}_bin'] = df_binned[col].clip(lower=0, upper=2)

    # Red count: range 0, 1, 2
    red_cols = ['home_red_count', 'away_red_count']
    for col in red_cols:
        if col in df_binned.columns:
            df_binned[f'{col}_bin'] = df_binned[col].clip(lower=0, upper=2)

    # Red diff: range -2 to 2
    if 'red_diff' in df_binned.columns:
        df_binned['red_diff_bin'] = df_binned['red_diff'].clip(lower=-2, upper=2)

    # Pre-match xP: rounding to the nearest .2 points
    xp_cols = ['pre_match_home_xP', 'pre_match_away_xP']
    for col in xp_cols:
        if col in df_binned.columns:
            # Multiply by 5, round, then divide by 5 gives nearest 0.2
            df_binned[f'{col}_bin'] = (df_binned[col] * 5).round() / 5

    # Yellows: range 0-6 or more
    yellow_cols = ['home_yellow_count', 'away_yellow_count']
    for col in yellow_cols:
        if col in df_binned.columns:
            df_binned[f'{col}_bin'] = df_binned[col].clip(lower=0, upper=6)

    return df_binned

def predict_outcome(test_point, historical_df, feature_cols, target_col, similarity_threshold=0.95):
    """
    Calculates similarity, filters historical data, and predicts outcome.
    Similarity is defined as the proportion of exactly matching bins/features.
    """
    # Calculate similarity index (percentage of matching features)
    # For numerical features, a distance metric could be used, but for categorical bins, exact match is standard.
    matches = (historical_df[feature_cols] == test_point[feature_cols]).sum(axis=1)
    similarity_index = matches / len(feature_cols)

    # Filter data with similarity index >= .95
    similar_samples = historical_df[similarity_index >= similarity_threshold]

    if len(similar_samples) == 0:
        return "No similar matches found", {'Home Win': 0, 'Draw': 0, 'Away Win': 0}, 0

    # Collate relative probabilities
    outcome_counts = similar_samples[target_col].value_counts(normalize=True)

    probs = {
        'Home Win': outcome_counts.get('Home Win', 0.0),
        'Draw': outcome_counts.get('Draw', 0.0),
        'Away Win': outcome_counts.get('Away Win', 0.0)
    }

    # Result with the highest probability
    prediction = max(probs, key=probs.get)

    return prediction, probs, len(similar_samples)

# --- Example Usage ---
# Assuming you have loaded your dataframe as `df` and it contains the required columns.
# df = prepare_binned_features(df)

# Define the columns that make up your features (the binned ones + time, match_type, time_1st_red)
# feature_columns = [
#     'score_diff_bin', 'time', 'home_attk_sub_bin', 'home_def_sub_bin',
#     'away_attk_sub_bin', 'away_def_sub_bin', 'home_red_count_bin',
#     'home_time_1st_red', 'away_red_count_bin', 'away_time_first_red',
#     'red_diff_bin', 'match_type', 'pre_match_home_xP_bin',
#     'pre_match_away_xP_bin', 'home_yellow_count_bin', 'away_yellow_count_bin'
# ]
# target_column = 'outcome_mapped'  # e.g., 'Home Win', 'Draw', 'Away Win'

# To test a datapoint:
# test_idx = 0
# test_data_point = df.iloc[test_idx]

# pred, prob_dict, sample_size = predict_outcome(test_data_point, df.drop(test_idx), feature_columns, target_column)
# print(f"Prediction: {pred}")
# print(f"Probabilities: {prob_dict}")
# print(f"Sample Size: {sample_size}")
